**Role in the paper.** Quality control and gap filling of the station data for one city with the
MetObs-toolkit (v1.1.1), for the urban climate network (`city_network`) and the meteorological service stations
(`meteo_stations`); the pipeline is described in SI Appendix, section 2.
Input: raw third-party data in `data/raw/observations/<City>/` (not redistributed, see README).
Output: cleaned series and station metadata in `data/processed/observations/<City>/<network>/`; the `_gapfree` subset
(stations without remaining gaps) is used in the analysis. Next step: `scripts/preprocessing/clean_obs_to_datasets.ipynb`.

In [ ]:
import metobs_toolkit
import matplotlib.pyplot as plt
import pandas as pd
import sys
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import datetime as dt
ROOT = Path.cwd().parent.parent.parent


In [ ]:
datapath = str(ROOT / 'data/raw/observations/Ghent/city_network/station_data')
dfs = {}
for file in Path(datapath).glob('*.csv'):
    station_id = file.stem.split('.')[0]
    print(f"Processing station {station_id}...")
    df = pd.read_csv(file, parse_dates=['DateTime'])
    df.set_index('DateTime', inplace=True)
    df = df.sort_index()
    df = df['Temperature']
    dfs[station_id] = df
df_tot = pd.concat(dfs.values(), axis=1, keys=dfs.keys())
df_tot.to_csv(str(ROOT / 'data/raw/observations/Ghent/city_network/Ghent_combined.csv'))


# City Network


In [ ]:
city = 'Ghent'
network= 'city_network'

data_dir = ROOT / f"data/raw/observations/{city}/{network}"

save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'
tables_dir = ROOT / f"results/tables/{city}/{network}"
plot_dir = ROOT / f"results/figures/{city}/{network}"
save_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)
save_pkl_dir.mkdir(parents=True, exist_ok=True)

## Loading data


In [ ]:
metobs_toolkit.build_template_prompt()

In [ ]:
# Define the paths to your files: 
# adapt these paths to your local setup
data_file = data_dir / f'{city}_combined.csv'
meta_data_file = data_dir /f'station_metadata_{city}.csv'
template = data_dir / 'template.json'

# Initiate a dataset:
dataset = metobs_toolkit.Dataset()

# Import your data :
dataset.import_data_from_file(
    input_data_file = data_file,
    input_metadata_file = meta_data_file,
    template_file = template,
    #The following arguments are filled with default values.
    freq_estimation_method = "median",
    freq_estimation_simplify_tolerance = "2min",
    origin_simplify_tolerance = "5min",
    timestamp_tolerance = "4min",
    kwargs_data_read = {},
    kwargs_metadata_read = {},
    templatefile_is_url = False,
)


In [ ]:
dataset.get_info()

In [ ]:
stations = dataset.metadf.index.get_level_values('name').to_list()

In [ ]:
#info from Google earht engine (GEE) for the station locations, see examples for setup and usage. 
dataset.get_LCZ()
dataset.get_altitude()
dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['LCZ'] ,save=True, filepath=f'{plot_dir}/LCZ_map_{city}_{network}', overwrite=True)
dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['worldcover'] ,save=True, filepath=f'{plot_dir}/Worldcover_map_{city}_{network}', overwrite=True)


In [ ]:
dataset.metadf

In [ ]:
%matplotlib widget
dataset.make_plot(obstype='temp', #Which observation type to plot. (See dataset.present_observations)
                  colorby='station', #if 'station', each station will be a different color
                  show_outliers=True,
                  show_gaps=True)


#### Save final products

In [ ]:
dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned.pkl', overwrite=True) #only save the subset of the dataset without unfilled gaps, to have a clean dataset for the next steps of the analysis.
dataset.to_csv(f'{save_dir}\\{city}_{network}_cleaned.csv', overwrite=True)

dataset.metadf.to_csv(f'{save_dir}\\{city}_{network}_metadata.csv')

### Analysis

In [ ]:

dataset_total = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned.pkl')
stations = dataset_total.metadf.index.get_level_values('name').to_list()

In [ ]:
analysis = metobs_toolkit.Analysis(Dataholder=dataset_total,)
analysis.get_info()

In [ ]:


print(f'The initial number of records: {analysis.df.shape[0]}')

#We can apply multiple consecutive filterings
analysis.apply_filter_on_records('season=="summer" | season=="autumn"') #Be aware of quotation! 


print(f'The number of records after filtering: {analysis.df.shape[0]}')

In [ ]:
analysis.plot_diurnal_cycle(obstype='temp', colorby='LCZ')

In [ ]:
%matplotlib widget
ax = analysis.plot_diurnal_cycle_with_reference_station(ref_station='melle',
                                                  obstype='temp',
                                                  colorby='name')


# Meteo data

In [ ]:

city= 'Ghent'
network= 'meteo_stations'
data_dir = ROOT / f"data/raw/observations/{city}/{network}"

save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'
tables_dir = ROOT / f"results/tables/{city}/{network}"
plot_dir = ROOT / f"results/figures/{city}/{network}"
save_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)
save_pkl_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.DataFrame()
for year in [2018, 2019 ,2020, 2021, 2022]: 
    for station in ['GHCNh_BEM00006434']:
        df_temp = pd.read_csv(data_dir / f'{station}_{year}.psv', sep='|')[['STATION', 'Station_name', 'DATE', 'temperature' ]]
        df = pd.concat([df, df_temp], axis=0)

df.to_csv(data_dir / f'{city}_meteo_data.csv', index=False)


In [ ]:
#metobs_toolkit.build_template_prompt()

In [ ]:
# Define the paths to your files: 
# adapt these paths to your local setup
data_file = data_dir / f'{city}_meteo_data.csv'
meta_data_file = data_dir /f'{city}_meteo_metadata.csv'
template = data_dir / 'template.json'

# Initiate a dataset:
meteo_dataset = metobs_toolkit.Dataset()

# Import your data :
meteo_dataset.import_data_from_file(
    input_data_file = data_file,
    input_metadata_file = meta_data_file,
    template_file = template,
    #The following arguments are filled with default values.
    freq_estimation_method = "median",
    freq_estimation_simplify_tolerance = "2min",
    origin_simplify_tolerance = "5min",
    timestamp_tolerance = "4min",
    kwargs_data_read = {},
    kwargs_metadata_read = {},
    templatefile_is_url = False,
)

In [ ]:
meteo_dataset.get_info()

the data of the station FIMU0002972 (= TURKU FINLAND , the rural ref at the airport) has timestamps at H:20:00 and H:50:00. to achiece synchronous data we resample this data to hourly data starting at '2018-01-01T00:00:00' with a 11m tolerance. This will set the H:50:00 data as hourly data but take in mind there is 10m advance...

In [ ]:
origin_datetime  = dt.datetime(2018, 1, 1, 0, 0, 0)


In [ ]:
meteo_dataset.resample(target_freq='1h', origin = origin_datetime,shift_tolerance= '11m') #

In [ ]:
%matplotlib widget
meteo_dataset.make_plot(obstype='temp', #Which observation type to plot. (See dataset.present_observations
                  colorby='label', #if 'station', each station will be a different color
                  show_outliers=True,
                  show_gaps=True)

In [ ]:
meteo_dataset.gap_overview_df()

In [ ]:
#info from Google earht engine (GEE) for the station locations, see examples for setup and usage. 
meteo_dataset.get_LCZ()
meteo_dataset.get_altitude()
meteo_dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['LCZ'] ,save=True, filepath=f'{plot_dir}/LCZ_map_{city}_{network}', overwrite=True)
meteo_dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['worldcover'] ,save=True, filepath=f'{plot_dir}/Worldcover_map_{city}_{network}', overwrite=True)


### Quality Control

In [ ]:

#1. gross value check
target = 'temp'
meteo_dataset.gross_value_check(
            obstype=target,
            lower_threshold=-30.0,
            upper_threshold=40.0)

#2. repetitions check
print(f'Applying repetitions check on {target} observations')
meteo_dataset.repetitions_check(
            obstype=target,
            max_N_repetitions=5 
)
#3. persistence check
print(f'Applying persistence check on {target} observations')
meteo_dataset.persistence_check(
            obstype=target,
            timewindow='5h', 
            min_records_per_window=3)

In [ ]:

#4. step check
print(f'Applying step check on {target} observations')
meteo_dataset.step_check(
            obstype=target,
            max_increase_per_second = 5.0 / 3600.0, #depends on standard unit! 
            max_decrease_per_second = -5.0 / 3600.0) #depends on standard unit! 

#5. window variation check
print(f'Applying window variation check on {target} observations')
meteo_dataset.window_variation_check(
            obstype=target,
            timewindow='4h',
            min_records_per_window=3,
            max_increase_per_second= 5.0 / 3600.0, #depends on standard unit! 
            max_decrease_per_second = -5.0 / 3600.0, #depends on standard unit! 
            )

In [ ]:
meteo_dataset.get_qc_stats(obstype='temp',make_plot=True)


In [ ]:
meteo_dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_after_basicQC.pkl', overwrite=True)

### Gap Filling

#### Interpolation

In [ ]:
meteo_dataset = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_after_basicQC.pkl')
amount_of_outliers_before = len(meteo_dataset.outliersdf)
print(f"Number of outliers before qc->gaps: {amount_of_outliers_before}")

In [ ]:
meteo_dataset.make_plot(obstype='temp', #Which observation type to plot. (See dataset.present_observations)
                    colorby='label', #if 'station', each station will be a different color
                    show_outliers=True,
                    show_gaps=True)

In [ ]:
meteo_dataset.convert_outliers_to_gaps(obstype='temp')

In [ ]:
#apply time(=linear but dt aware) interpolation for small gaps (up to 2 consecutive missing values)
meteo_dataset.interpolate_gaps(
                    obstype='temp', #Which gaps to fill
                    overwrite_fill = False, #Overwrite previous filled values if they are present
                    method='time', #which interpolation method
                    max_gap_duration_to_fill=pd.Timedelta("4h")
                    )

In [ ]:

#apply polynomial interpolation for larger gaps (up to 5 consecutive missing values)
meteo_dataset.interpolate_gaps(
                obstype='temp',
                method='polynomial',
                overwrite_fill=False, #the gaps that are already filled are skipped (i.g. chainig GF methods)
                n_leading_anchors=3, #at least 3 leading anchors are needed for 3th order polynomial interpolation
                n_trailing_anchors=3, #at least 3 trailing anchors are needed for 3th order polynomial interpolation
                max_gap_duration_to_fill=pd.Timedelta("8h"), #the maximum number of consecutive timestamps to fill. The result is dependent on the time-resolution of the gap (=equal to that of the related SensorData). Defaults to 10
                max_lead_to_gap_distance='24h', #the maximum distance (in time) beween the leading anchors and the start of the gap.
                max_trail_to_gap_distance = '24h',
                method_kwargs={'order':3}, #all extra arguments to pass to the pandas.Dataframe.interpolate method.
)


In [ ]:
meteo_dataset.make_plot(obstype='temp', #Which observation type to plot. (See dataset.present_observations)
                    colorby='label', #if 'station', each station will be a different color
                    show_outliers=True,
                    show_gaps=True)    

In [ ]:
meteo_dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_after_interpolation_gf.pkl', overwrite=True) #set overwrite to True to overwrite if the file already exists, otherwise an error is raised.

#### fill gaps with model data

In [ ]:
meteo_dataset = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_after_interpolation_gf.pkl')

Remaining gaps are filled with ERA5-Land, debiased with the station's own observations around each gap (MetObs-toolkit).

In [ ]:
"""era5_manager = metobs_toolkit.default_GEE_datasets['ERA5-land']

#Specify a start and end timestamp
tstart = pd.Timestamp('2018-01-01 00:00:00')
tend = pd.Timestamp('2022-12-01 00:00:00')

#Extract the timeseries
era5_temp = meteo_dataset.get_gee_timeseries_data(
            gee_dynamic_manager=era5_manager, #The datasetmanager to use
            startdt_utc=tstart,
            enddt_utc=tend,
            obstypes=['temp'], #the observationtypes to extract, must be knonw modelobstypes
            get_all_bands=False, #If true, all bands are extracted (but not stored in the stations if the band is unknwown)
            )
"""

In [ ]:
era5_manager = metobs_toolkit.default_GEE_datasets['ERA5-land']

ERA5_data = f"{data_dir}\\ERA5-land_timeseries_data_{city}.csv"
#I added the file with the ERA5 data to the datafolder so you should just adapt the path to your dir, but try to download yourself based on the example scripts

meteo_dataset.import_gee_data_from_file( filepath= ERA5_data,  gee_dynamic_manager=era5_manager  )


In [ ]:
meteo_dataset.fill_gaps_with_weighted_diurnal_debiased_modeldata(
            obstype='temp',
            leading_period_duration=pd.Timedelta('10d'),
            trailing_period_duration=pd.Timedelta('10d'),
            min_lead_debias_sample_size=2,
            min_trail_debias_sample_size=2,
            max_gap_duration_to_fill=pd.Timedelta('33d'))


In [ ]:
gaps = meteo_dataset.gap_overview_df()
remaining_gaps = gaps[gaps['label']=='failed gapfill']
remaining_gaps

In [ ]:
meteo_dataset.fill_gaps_with_debiased_modeldata(
            obstype='temp',
            overwrite_fill=False,
            leading_period_duration=pd.Timedelta('1h'),
            trailing_period_duration=pd.Timedelta('7d'),
            min_leading_records_total=0,
            min_trailing_records_total=5,
            max_gap_duration_to_fill=pd.Timedelta('6d'))
meteo_dataset.fill_gaps_with_debiased_modeldata(
            obstype='temp',
            overwrite_fill=False,
            leading_period_duration=pd.Timedelta('7d'),
            trailing_period_duration=pd.Timedelta('1h'),
            min_leading_records_total=5,
            min_trailing_records_total=0,
            max_gap_duration_to_fill=pd.Timedelta('6d'))

In [ ]:
gaps = meteo_dataset.gap_overview_df()
remaining_gaps = gaps[gaps['label']=='failed gapfill']
remaining_gaps

In [ ]:
%matplotlib widget
meteo_dataset.make_plot(obstype='temp', #Which observation type to plot. (See dataset.present_observations)
                    colorby='label', #if 'station', each station will be a different color
                    show_outliers=True,
                    show_gaps=True)

In [ ]:
meteo_dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned.pkl', overwrite=True) #set overwrite to True to overwrite if the file already exists, otherwise an error is raised.
meteo_dataset.to_csv(f'{save_dir}\\{city}_{network}_cleaned.csv', overwrite=True)
meteo_dataset.metadf.to_csv(f'{save_dir}\\{city}_{network}_metadata.csv')

# Combine meteo and city network datasets

In [ ]:
subset = ''

In [ ]:
city = 'Ghent'
network= 'city_network'

data_dir = ROOT / f"data/raw/observations/{city}/{network}"
save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'

dataset = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned.pkl')

In [ ]:
city = 'Ghent'
network= 'meteo_stations'

data_dir = ROOT / f"data/raw/observations/{city}/{network}"
save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'

meteo_dataset = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_meteo_stations_cleaned.pkl')

In [ ]:
city = 'Ghent'
network= 'combined_network'

data_dir = ROOT / f"data/raw/observations/{city}/{network}"

save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'
tables_dir = ROOT / f"results/tables/{city}/{network}"
plot_dir = ROOT / f"results/figures/{city}/{network}"
save_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)
save_pkl_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
combi_dataset = dataset+meteo_dataset

In [ ]:
#info from Google earht engine (GEE) for the station locations, see examples for setup and usage. 
combi_dataset.get_LCZ()
combi_dataset.get_altitude()
combi_dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['LCZ'] ,save=True, filepath=f'{plot_dir}/LCZ_map_{city}_{network}_{subset}', overwrite=True)
combi_dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['worldcover'] ,save=True, filepath=f'{plot_dir}/Worldcover_map_{city}_{network}_{subset}', overwrite=True)


In [ ]:
combi_dataset.metadf

In [ ]:
combi_dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned.pkl', overwrite=True) #set overwrite to True to overwrite if the file already exists, otherwise an error is raised.
combi_dataset.to_csv(f'{save_dir}\\{city}_{network}_cleaned.csv', overwrite=True)
combi_dataset.metadf.to_csv(f'{save_dir}\\{city}_{network}_metadata.csv')

In [ ]:
table_save_dir = ROOT / f"results/tables/{city}"
table_save_dir.mkdir(parents=True, exist_ok=True)
wc = combi_dataset.get_landcover_fractions(buffers = [10,50,100,200], aggregate=False)
wc = wc.replace(np.nan,0)
wc['rural_fraction'] = wc['Grassland'] + wc['Cropland'] + wc['Bare / sparse vegetation'] + wc['Tree cover']
wc['urban_fraction'] = wc['Built-up'] 
wc.to_csv(f'{table_save_dir}/landcover_fractions_{city}_{network}.csv', index=True)

In [ ]:
wc